# 09 — Qdrant Vector Database

**FinRAG learning series · Notebook 9 of 14**

## Learning objectives
1. Understand what a vector database adds to a list of vectors (indexes, payloads, filters, persistence).
2. Create a collection, insert points with payloads, search, filter and scroll in **Qdrant**.
3. Read FinRAG's real collection configuration and run production-style filtered searches.
4. See how payload filters enforce **tenant isolation** and document selection.

## Prerequisites
Notebook 08. Qdrant running at the URL configured in `.env` (default `http://localhost:6333`). This notebook creates a **temporary
demo collection** and deletes it at the end; the production collection is only read.

## Concept
Searching 1,000 vectors by brute force (notebook 08) is fine; searching millions per query is not. Qdrant stores vectors in an
**HNSW graph** index for fast approximate nearest-neighbour search, keeps a JSON **payload** with each point, and can **filter** on
payload fields during the search.

FinRAG uses one collection `finrag_chunks`:
* point id = `document_chunks.id` in PostgreSQL (so every hit maps back to text and location);
* vector = 384-d BGE embedding (cosine);
* payload = `owner_id`, `document_id`, `chunk_index`, `file_format`, `filename` (keyword indexes on the first three filters).

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


In [2]:
from qdrant_client.http import models as qm
from app.rag import vector_store, embeddings
client = vector_store.client()
print(nb.safe_settings("qdrant_url", "qdrant_collection"))
print("collections:", [c.name for c in client.get_collections().collections])

{'qdrant_url': 'http://localhost:6333', 'qdrant_collection': 'finrag_chunks'}
collections: ['finrag_chunks']


## 1. The production collection (read-only)

In [3]:
from app.core.config import get_settings
info = client.get_collection(get_settings().qdrant_collection)
print("status:", info.status, "| points:", info.points_count)
print("vectors:", info.config.params.vectors)
print("payload indexes:", {k: str(v.data_type) for k, v in (info.payload_schema or {}).items()})

status: green | points: 1578
vectors: size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None
payload indexes: {'file_format': 'keyword', 'document_id': 'keyword', 'owner_id': 'keyword'}


## 2. A temporary demo collection

In [4]:
import uuid
DEMO = f"nb_demo_{uuid.uuid4().hex[:8]}"
client.create_collection(DEMO, vectors_config=qm.VectorParams(size=embeddings.embedding_dim(), distance=qm.Distance.COSINE))
client.create_payload_index(DEMO, field_name="page", field_schema=qm.PayloadSchemaType.INTEGER)
print("created", DEMO)

created nb_demo_19a8a781


Insert 40 real chunks (text from the database, vectors from the configured model) with their page in the payload:

In [5]:
from app.models import Document, DocumentChunk
from app.document_processing.elements import describe_location
db = nb.db_session()
user = nb.notebook_user(db)
doc = (db.query(Document).filter(Document.owner_id == user.id, Document.file_format == "pdf", Document.status == "processed")
       .order_by(Document.chunk_count.desc()).first())
chunks = db.query(DocumentChunk).filter_by(document_id=doc.id).order_by(DocumentChunk.chunk_index).limit(400).all()[::10]
vecs = embeddings.embed_documents([c.text for c in chunks])
client.upsert(DEMO, points=[qm.PointStruct(id=str(c.id), vector=v,
              payload={"page": c.location.get("page"), "kind": ",".join(c.element_types), "preview": c.text[:80]})
              for c, v in zip(chunks, vecs)], wait=True)
print("points in demo collection:", client.count(DEMO, exact=True).count)

points in demo collection: 40


## 3. Search, then search with a filter

In [6]:
q = embeddings.embed_query("consolidated balance sheet total assets")
res = client.query_points(DEMO, query=q, limit=3, with_payload=True).points
for p in res:
    print(f"{p.score:.3f} page {p.payload['page']:>3} | {p.payload['preview']!r}")
print("--- only pages 1-40")
flt = qm.Filter(must=[qm.FieldCondition(key="page", range=qm.Range(gte=1, lte=40))])
for p in client.query_points(DEMO, query=q, query_filter=flt, limit=3, with_payload=True).points:
    print(f"{p.score:.3f} page {p.payload['page']:>3} | {p.payload['preview']!r}")

0.718 page  75 | '[Context: to the Standalone Financial Statements for the year ended 31st March, '
0.674 page  73 | 'Notes\nto the Standalone Financial Statements for the year ended 31st March, 2024'
0.667 page  71 | 'Balance Sheet based on Current/ Non-Current which comprises initial cost of the '
--- only pages 1-40
0.663 page   7 | '[Context: • Strong balance sheet with D7,42,922 Crore net worth our stakeholders'
0.627 page  39 | 'Debenture Trustee Axis Trustee Services Limited\nThe Ruby, 2nd Floor, SW, 29, Sen'
0.603 page  33 | 'in dealing with all the stakeholders. and has a zero-tolerance approach to −Inde'


## 4. Scroll: read points without a query

In [7]:
pts, next_offset = client.scroll(DEMO, limit=3, with_payload=True, with_vectors=False)
for p in pts:
    print(p.id, p.payload["page"], p.payload["kind"])
print("next page offset:", next_offset)

0036fb50-d6bf-442f-96c8-1c669eb42799 51 heading,paragraph
1a33eb6a-72af-4ecc-989c-5878bceec452 15 paragraph,table_row
1f284084-3eb7-4a3c-817a-c87db77316ba 20 paragraph
next page offset: 2058ab0e-462a-4d91-a3ec-20f53d2a0de2


In [8]:
client.delete_collection(DEMO)
print("deleted", DEMO, "| remaining collections:", [c.name for c in client.get_collections().collections])

deleted nb_demo_19a8a781 | remaining collections: ['finrag_chunks']


## 5. Production-style search with tenant isolation
`vector_store.search` always filters on `owner_id` **and** the selected `document_id`s. Searching the same vector with another
owner id returns nothing — the database cannot leak one user's documents into another user's answers.

In [9]:
qv = embeddings.embed_query("How much debt does the company have?")
hits = vector_store.search(qv, str(user.id), [str(doc.id)], limit=3)
for h in hits:
    c = db.get(DocumentChunk, uuid.UUID(str(h.id)))
    print(f"{h.score:.3f} | {describe_location(c.location):<28} | {nb.shorten(c.text, 90)}")
print("same query, a different owner id ->", len(vector_store.search(qv, str(uuid.uuid4()), [str(doc.id)], limit=3)), "hits")
db.close()

0.692 | page 98 (printed p. 192)     | [Context: to the Standalone Financial Statements for the year ended 31st March, 2024 | Th…
0.691 | page 103 (printed p. 202)    | [Context: to the Standalone Financial Statements for the year ended 31st March, 2024 | (C…
0.686 | page 98 (printed p. 192)     | [Context: to the Standalone Financial Statements for the year ended 31st March, 2024 | Th…
same query, a different owner id -> 0 hits


## Interpretation
* The payload filter narrowed results to the requested pages *inside* the vector search — not by post-filtering.
* Every production hit maps back to a PostgreSQL chunk with its exact location, which becomes the citation.
* Tenant isolation is enforced by the filter (and re-checked in SQL by the retriever).

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| `Connection refused` | Qdrant not running | Start it (`deployment/dev-services.sh` or docker compose) |
| Version warning client/server | Client much newer than server | Keep versions within one minor release |
| `Wrong input: Vector dimension error` | Model changed | Recreate collection; reprocess |
| Filter returns nothing | Field not indexed / wrong type | Create a payload index with the right schema |

## Practical exercises
1. Add a `file_format` filter to the demo search and test it with chunks from a spreadsheet document.
2. Compare search latency with and without a payload filter on the production collection.
3. Read the Qdrant snapshot API and run `deployment/backup.sh` in a test environment.

## Summary
You inspected the production collection, created and searched a temporary one with payload filters, and saw how FinRAG's filtered search
keeps users' documents separate.

## Further learning
* Qdrant documentation: https://qdrant.tech/documentation/
* HNSW paper: https://arxiv.org/abs/1603.09320
* FinRAG source: `backend/app/rag/vector_store.py`
* Next: **10 — The RAG pipeline**.